# Workflow for a multi-regional energy system

In this application of the ETHOS.FINE framework, a multi-regional energy system is modeled and optimized.

All classes which are available to the user are utilized and examples of the selection of different parameters within these classes are given.

The workflow is structures as follows:
1. Required packages are imported and the input data path is set
2. An energy system model instance is created
3. Commodity sources are added to the energy system model
4. Commodity conversion components are added to the energy system model
5. Commodity storages are added to the energy system model
6. Commodity transmission components are added to the energy system model
7. Commodity sinks are added to the energy system model
8. The energy system model is optimized
9. Selected optimization results are presented


# 1. Import required packages and set input data path

The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [1]:
import fine as fn
import matplotlib.pyplot as plt
from getData import getData
import pandas as pd
import os

cwd = os.getcwd()
data = getData()

%matplotlib inline
%load_ext autoreload
%autoreload 2

# 2. Create an energy system model instance 

The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [2]:
locations = {"GermanyRegion", "FranceRegion"}
commodityUnitsDict = {
    "electricity": r"GW$_{el}$",
    "methane": r"GW$_{CH_{4},LHV}$",
    "biogas": r"GW$_{biogas,LHV}$",
    "CO2": r"Mio. t$_{CO_2}$/h",
    "hydrogen": r"GW$_{H_{2},LHV}$",
}
commodities = {"electricity", "hydrogen", "methane", "biogas", "CO2"}
materials = {
    "steel", "copper",
    "windonshore_steel_scrap", "windoffshore_steel_scrap",
    "windonshore_copper_scrap", "windoffshore_copper_scrap"
}

materialUnitsDict = {
    "steel": "tons",
    "copper": "tons",
    "windonshore_steel_scrap": "tons",
    "windoffshore_steel_scrap": "tons",
    "windonshore_copper_scrap": "tons",
    "windoffshore_copper_scrap": "tons",
}

numberOfTimeSteps = 8760
hoursPerTimeStep = 1


#pathwayBalanceLimit = pd.DataFrame(columns=["GermanyRegion", "FranceRegion","Total","lowerBound"], index=["Copper_Resources", "Steel_Resources", "Lithium_Resources"])

#pathwayBalanceLimit.loc["Copper_Resources"] = [500000, 500000,None, False]
#pathwayBalanceLimit.loc["Steel_Resources"] = [500000, 500000,None, False]
#pathwayBalanceLimit.loc["Lithium_Resources"] = [500000, 500000,None, False]



# pathwayBalanceLimit.loc["Copper_Resources"] = [5555000, 5555000, False]
# pathwayBalanceLimit.loc["Steel_Resources"] = [5555000, 5555000, False]
# pathwayBalanceLimit.loc["Lithium_Resources"] = [5555000, 5555000, False]


In [3]:
numberOfInvestmentPeriods=3
startYear=2020
interval=5

In [4]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    materials=materials,
    numberOfInvestmentPeriods=numberOfInvestmentPeriods,
    startYear=startYear, 
    investmentPeriodInterval=interval,
    numberOfTimeSteps=8760,
    commodityUnitsDict=commodityUnitsDict,
    materialUnitsDict=materialUnitsDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
   # pathwayBalanceLimit=pathwayBalanceLimit,
)

DEBUG: type(self.materialUnitsDict) = <class 'dict'>
DEBUG: self.materialUnitsDict = {'steel': 'tons', 'copper': 'tons', 'windonshore_steel_scrap': 'tons', 'windoffshore_steel_scrap': 'tons', 'windonshore_copper_scrap': 'tons', 'windoffshore_copper_scrap': 'tons'}
DEBUG: type(self.materials) = <class 'set'>
DEBUG: self.materials = {'steel', 'windonshore_copper_scrap', 'windonshore_steel_scrap', 'copper', 'windoffshore_copper_scrap', 'windoffshore_steel_scrap'}


In [5]:
CO2_reductionTarget = 1

# 3. Add commodity sources to the energy system model

## 3.1. Electricity sources

### Wind onshore

In [6]:
esM.add(
    fn.Source(
        esM=esM,
        name="windonshore",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=1.1,
        opexPerCapacity=1.1 * 0.02,
        interestRate=0.08,
        economicLifetime=5,
        materialIntensity = {
            'GermanyRegion': {
                'steel': pd.Series({  2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64'),
                'copper': pd.Series({  2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64')
 
            },
             'FranceRegion': {
                'steel': pd.Series({ 2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64'),
                'copper': pd.Series({ 2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64')
               
            }
        },
        materialRecovery = {
            'GermanyRegion': {
                'steel': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64'),
                'copper': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            },
            'FranceRegion': {
                'steel': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64'),
                'copper': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            }
        }
        
    )
)

Full load hours:

In [7]:
data["Wind (onshore), operationRateMax"].sum()

FranceRegion     2350.292663
GermanyRegion    1572.003960
dtype: float64

### Wind offshore

In [8]:
esM.add(
    fn.Source(
        esM=esM,
        name="windoffshore",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (offshore), operationRateMax"],
        capacityMax=data["Wind (offshore), capacityMax"],
        investPerCapacity=2.3,
        opexPerCapacity=2.3 * 0.02,
        interestRate=0.08,
        economicLifetime=5,
        materialIntensity = {
            'GermanyRegion': {
                'steel': pd.Series({  2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64'),
                'copper': pd.Series({  2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64')
 
            },
             'FranceRegion': {
                'steel': pd.Series({ 2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64'),
                'copper': pd.Series({ 2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64')
               
            }
        },
        materialRecovery = {
            'GermanyRegion': {
                'steel': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64'),
                'copper': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            },
            'FranceRegion': {
                'steel': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64'),
                'copper': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            }
        }
         
    )
)

Full load hours:

In [9]:
data["Wind (offshore), operationRateMax"].sum()

FranceRegion     4301.655834
GermanyRegion    4435.420314
dtype: float64

### PV

In [10]:
esM.add(
    fn.Source(
        esM=esM,
        name="PV",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["PV, operationRateMax"],
        capacityMax=data["PV, capacityMax"],
        investPerCapacity=0.65,
        opexPerCapacity=0.65 * 0.02,
        interestRate=0.08,
        economicLifetime=25,
    )
)

Full load hours:

In [11]:
data["PV, operationRateMax"].sum()

FranceRegion     1053.579422
GermanyRegion    1113.216464
dtype: float64

### Exisisting run-of-river hydroelectricity plants

In [12]:
esM.add(
    fn.Source(
        esM=esM,
        name="Existing run-of-river plants",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateFix=data["Existing run-of-river plants, operationRateFix"],
        tsaWeight=0.01,
        capacityFix=data["Existing run-of-river plants, capacityFix"],
        investPerCapacity=0,
        opexPerCapacity=0.208,
    )
)

## 3.2. Methane (natural gas and biogas)

### Natural gas

In [13]:
esM.add(
    fn.Source(
        esM=esM,
        name="Natural gas purchase",
        commodity="methane",
        hasCapacityVariable=False,
        commodityCost=0.0331 * 1e-3,
    )
)

### Biogas

In [14]:
esM.add(
    fn.Source(
        esM=esM,
        name="Biogas purchase",
        commodity="biogas",
        operationRateMax=data["Biogas, operationRateMax"],
        hasCapacityVariable=False,
        commodityCost=0.05409 * 1e-3,
    )
)

# 4. Add conversion components to the energy system model

### Combined cycle gas turbine plants

In [15]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="CCGT plants (methane)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "electricity": 1,
            "methane": -1 / 0.6,
            "CO2": 201 * 1e-6 / 0.6,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.65,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=5,
        materialIntensity = {
            'GermanyRegion': {
                'steel': pd.Series({  2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64'),
                'copper': pd.Series({  2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64')
 
            },
             'FranceRegion': {
                'steel': pd.Series({ 2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64'),
                'copper': pd.Series({ 2015: 1.1, 2020: 3.1, 2025: 3.0, 2030: 2.9}, dtype='float64')
               
            }
        },
        materialRecovery = {
            'GermanyRegion': {
                'steel': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64'),
                'copper': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            },
            'FranceRegion': {
                'steel': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64'),
                'copper': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            }
        }
    )
)

### New combined cycle gas turbine plants for biogas

In [16]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="New CCGT plants (biogas)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": 1, "biogas": -1 / 0.63},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### New combined cycly gas turbines for hydrogen

In [17]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="New CCGT plants (hydrogen)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": 1, "hydrogen": -1 / 0.63},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### Electrolyzers

In [18]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electrolyzer",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=10,
    )
)

### rSOC

In [19]:
capexRSOC = 1.5

esM.add(
    fn.Conversion(
        esM=esM,
        name="rSOEC",
        physicalUnit=r"GW$_{el}$",
        linkedConversionCapacityID="rSOC",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.6},
        hasCapacityVariable=True,
        investPerCapacity=capexRSOC / 2,
        opexPerCapacity=capexRSOC * 0.02 / 2,
        interestRate=0.08,
        economicLifetime=10,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="rSOFC",
        physicalUnit=r"GW$_{el}$",
        linkedConversionCapacityID="rSOC",
        commodityConversionFactors={"electricity": 1, "hydrogen": -1 / 0.6},
        hasCapacityVariable=True,
        investPerCapacity=capexRSOC / 2,
        opexPerCapacity=capexRSOC * 0.02 / 2,
        interestRate=0.08,
        economicLifetime=10,
    )
)

# 5. Add commodity storages to the energy system model

## 5.1. Electricity storage

### Lithium ion batteries

The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [ ]:
esM.add(
    fn.Storage(
        esM=esM,
        name="battery",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=5,
        materialIntensity = {
            'GermanyRegion': {
                'lithium': pd.Series({  2015: 3.1, 2020: 2.1, 2025: 1.0, 2030: 1.9}, dtype='float64')
 
            },
             'FranceRegion': {
                'lithium': pd.Series({  2015: 3.1, 2020: 2.1, 2025: 1.0, 2030: 1.9}, dtype='float64')
               
            }
        },
        materialRecovery = {
            'GermanyRegion': {
                'lithium': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            },
            'FranceRegion': {
                'lithium': pd.Series({2020: 1, 2025: 0.85, 2030: 0.9}, dtype='float64')
            }
        }
         
    )
)

## 5.2. Hydrogen storage

### Hydrogen filled salt caverns
The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

In [21]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (hydrogen)",
        commodity="hydrogen",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=133,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (hydrogen), capacityMax"],
        investPerCapacity=0.00011,
        opexPerCapacity=0.00057,
        interestRate=0.08,
        economicLifetime=30,
    )
)

## 5.3. Methane storage

### Methane filled salt caverns

In [22]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (biogas)",
        commodity="biogas",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=443,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (methane), capacityMax"],
        investPerCapacity=0.00004,
        opexPerCapacity=0.00001,
        interestRate=0.08,
        economicLifetime=30,
    )
)

## 5.4 Pumped hydro storage

### Pumped hydro storage

In [23]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Pumped hydro storage",
        commodity="electricity",
        chargeEfficiency=0.88,
        dischargeEfficiency=0.88,
        hasCapacityVariable=True,
        selfDischarge=1 - (1 - 0.00375) ** (1 / (30 * 24)),
        chargeRate=0.16,
        dischargeRate=0.12,
        capacityFix=data["Pumped hydro storage, capacityFix"],
        investPerCapacity=0,
        opexPerCapacity=0.000153,
    )
)

# 6. Add commodity transmission components to the energy system model

## 6.1. Electricity transmission

### AC cables

esM.add(fn.LinearOptimalPowerFlow(esM=esM, name='AC cables', commodity='electricity',
                                  hasCapacityVariable=True, capacityFix=data['AC cables, capacityFix'],
                                  reactances=data['AC cables, reactances']))

In [24]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="AC cables",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityFix=data["AC cables, capacityFix"],
    )
)

The distances of a component are set to a normalized value of 1.


### DC cables

In [25]:
# esM.add(
#     fn.Transmission(
#         esM=esM,
#         name="DC cables",
#         commodity="electricity",
#         losses=data["DC cables, losses"],
#         distances=data["DC cables, distances"],
#         hasCapacityVariable=True,
#         capacityFix=data["DC cables, capacityFix"],
#     )
# )

## 6.2 Methane transmission

### Methane pipeline

In [26]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="Pipelines (biogas)",
        commodity="biogas",
        distances=data["Pipelines, distances"],
        hasCapacityVariable=True,
        hasIsBuiltBinaryVariable=True,
        bigM=300,
        locationalEligibility=data["Pipelines, eligibility"],
        capacityMax=data["Pipelines, eligibility"] * 15,
        sharedPotentialID="pipelines",
        investPerCapacity=0.000037,
        investIfBuilt=0.000314,
        interestRate=0.08,
        economicLifetime=40,
    )
)

## 6.3 Hydrogen transmission

### Hydrogen pipelines

In [27]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="Pipelines (hydrogen)",
        commodity="hydrogen",
        distances=data["Pipelines, distances"],
        hasCapacityVariable=True,
        hasIsBuiltBinaryVariable=True,
        bigM=300,
        locationalEligibility=data["Pipelines, eligibility"],
        capacityMax=data["Pipelines, eligibility"] * 15,
        sharedPotentialID="pipelines",
        investPerCapacity=0.000177,
        investIfBuilt=0.00033,
        interestRate=0.08,
        economicLifetime=40,
    )
)

# 7. Add commodity sinks to the energy system model

## 7.1. Electricity sinks

### Electricity demand

In [28]:
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=data["Electricity demand, operationRateFix"],
    )
)

## 7.2. Hydrogen sinks

### Fuel cell electric vehicle (FCEV) demand

In [29]:
FCEV_penetration = 0.5
esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=data["Hydrogen demand, operationRateFix"] * FCEV_penetration,
    )
)

## 7.3. CO2 sinks

### CO2 exiting the system's boundary

In [30]:
esM.add(
    fn.Sink(
        esM=esM,
        name="CO2 to enviroment",
        commodity="CO2",
        hasCapacityVariable=False,
        commodityLimitID="CO2 limit",
        yearlyLimit=366 * (1 - CO2_reductionTarget),
    )
)

In [31]:
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        hasCapacityVariable=False,
        commodity="steel",
        material=True,      
    )
)

 


In [ ]:
esM.add(
    fn.Source(
        esM=esM,
        name="Steel supply",
        hasCapacityVariable=False,
        commodity="steel"
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Copper supply",
        hasCapacityVariable=False,
        commodity="copper"
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="Lithium supply",
        hasCapacityVariable=False,
        commodity="lithium"
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="windonshore_steel_scrap",
        commodity="windonshore_steel_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="windoffshore_steel_scrap",
        commodity="windoffshore_steel_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="windonshore_copper_scrap",
        commodity="windonshore_copper_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="windoffshore_copper_scrap",
        commodity="windoffshore_copper_scrap",
        hasCapacityVariable=False,
        material=True
    )
)



esM.add(
    fn.Source(
        esM=esM,
        name="battery_lithium_scrap",
        commodity="battery_lithium_scrap",
        hasCapacityVariable=False,
        material=True
    )
)

In [ ]:
# esM.add(
#     fn.Conversion(
#         esM=esM,
#         name="Lithium Recycler",
#         physicalUnit=r"GW$_{el}$",
#         commodityConversionFactors={"lithium_scrap": -1, "lithium":0.4},
#         hasCapacityVariable=True,
#         investPerCapacity=0.7,
#         opexPerCapacity=0.021,
#         interestRate=0.08,
#         economicLifetime=33,
#     )
# )

: 

In [ ]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Steel Recycler Onshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"windonshore_steel_scrap": -1, "steel":0.4},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)
esM.add(
    fn.Conversion(
        esM=esM,
        name="Steel Recycler Offshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"windoffshore_steel_scrap": -1, "steel":0.4},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)
esM.add(
    fn.Conversion(
        esM=esM,
        name="Copper Recycler Onshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"windonshore_copper_scrap": -1, "copper":0.4},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)
esM.add(
    fn.Conversion(
        esM=esM,
        name="Copper Recycler Offshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"windoffshore_copper_scrap": -1, "copper":0.4},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)


esM.add(
    fn.Conversion(
        esM=esM,
        name="Copper Recycler Battery",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"battery_lithium_scrap": -1, "lithium":0.5},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

In [34]:
esM.generationMaterialSinks()

Existing material sinks: {'steel'}
Missing materials sinks: {'windoffshore_copper_scrap', 'windoffshore_steel_scrap', 'windonshore_copper_scrap', 'copper', 'windonshore_steel_scrap'}
New sink added: Windoffshore_copper_scrap demand
New sink added: Windoffshore_steel_scrap demand
New sink added: Windonshore_copper_scrap demand
New sink added: Copper demand
New sink added: Windonshore_steel_scrap demand


All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

# 8 Temporal Aggregation

In [44]:
esM.aggregateTemporally(numberOfTypicalPeriods=10)


Clustering time series data with 10 typical periods and 24 time steps per period 
further clustered to 12 segments per period...
		(12.3721 sec)



### Optimization

In [36]:
esM.optimize(timeSeriesAggregation=True, solver="gurobi")

Time series aggregation specifications:
Number of typical periods:3, number of time steps per period:24, number of segments per period:12

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.1425 sec)

Declaring sets, variables and constraints for ConversionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0604 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(3.1145 sec)

Declaring sets, variables and constraints for TransmissionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.0231 sec)

Declaring shared potential constraint...
		(0.0009 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.0933 sec)

Declaring material balances...
('FranceRegion', 'Steel demand', 'steel', 0)
('FranceRegion'

/fast/home/n-ludwig/model_Git/fine/fine/storage.py:2090: UserWarning: Charge and discharge at the same time for component Salt caverns (biogas)
  warnings.warn(


for StorageModel ...       (2.0342sec)
for TransmissionModel ...  (0.9518sec)
		(5.9446 sec)



In [38]:
print("Scrap Set Entries:")
for entry in esM.pyM.materialScrapSet:
    print(entry)


Scrap Set Entries:
('FranceRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 0)
('FranceRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 1)
('FranceRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 2)
('GermanyRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 0)
('GermanyRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 1)
('GermanyRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 2)
('FranceRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 0)
('FranceRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 1)
('FranceRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 2)
('GermanyRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 0)
('GermanyRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 1)
('GermanyRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 2)
('FranceRegion', 'windonshore_copper_scrap', 'windonshore', 'copper', 0)
('FranceRegion', 'windonshore_copper_s

In [39]:
print("Scrap Set Entries:")
for entry in esM.pyM.materialScrapSet:
    print(entry)


Scrap Set Entries:
('FranceRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 0)
('FranceRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 1)
('FranceRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 2)
('GermanyRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 0)
('GermanyRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 1)
('GermanyRegion', 'windonshore_steel_scrap', 'windonshore', 'steel', 2)
('FranceRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 0)
('FranceRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 1)
('FranceRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 2)
('GermanyRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 0)
('GermanyRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 1)
('GermanyRegion', 'windoffshore_steel_scrap', 'windoffshore', 'steel', 2)
('FranceRegion', 'windonshore_copper_scrap', 'windonshore', 'copper', 0)
('FranceRegion', 'windonshore_copper_s

In [ ]:
print("Defined Source components:")
for name, comp in esM.componentModelingDict["SourceSinkModel"].componentsDict.items():
    print(name, comp.commodity, getattr(comp, "material", False))


: 

: 

In [ ]:
print("Decommissioning values (nonzero):")
import pyomo.environ as pyomo
for (loc, compName, ip), var in esM.pyM.decommis_srcSnk.items():
    val = pyomo.value(var)
    if val > 0 and ("windonshore" in compName or "windoffshore" in compName):
        print(f"{compName} @ {loc}, IP {ip}: {val}")


: 

: 

In [ ]:
print("processedMaterialRecovery:", comp.processedMaterialRecovery)


: 

: 

In [ ]:
print(esM.componentModelingDict["SourceSinkModel"].componentsDict.keys())


: 

: 

In [ ]:
print((loc, "Wind (onshore)", ip) in esM.pyM.decommis_srcSnk)


: 

: 

In [ ]:
print(comp.economicLifetime)


: 

: 

In [ ]:
print("Scrap Set Entries:")
for entry in esM.pyM.materialScrapSet:
    print(entry)


: 

: 

# 9. Selected results output

Plot locations (GeoPandas required)

In [ ]:
# Import the geopandas package for plotting the locations
import geopandas as gpd

: 

: 

### Sources and Sink

Show optimization summary

In [45]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2, ip=2025)

FranceRegion  \
Component                    Property        Unit                                    
Biogas purchase              NPVcontribution [1e9 Euro]                   0.500249   
                             TAC             [1e9 Euro/a]                 0.170456   
                             commodCosts     [1e9 Euro/a]                 0.170456   
                             operation       [GW$_{biogas,LHV}$*h/a]   3151.348212   
                                             [GW$_{biogas,LHV}$*h]     3151.348212   
Copper demand                operation       [tons/a]                   209.754434   
Copper supply                operation       [tons/a]                    136.06071   
Electricity demand           operation       [GW$_{el}$*h/a]          66115.987455   
                                             [GW$_{el}$*h]            66115.987455   
Existing run-of-river plants NPVcontribution [1e9 Euro]                   0.022864   
                             TAC             [1e9 Euro/a]                 0.007791   
                             capacity        [GW$_{el}$]                  0.037456   
                             operation       [GW$_{el}$*h/a]            170.988002   
                                             [GW$_{el}$*h]              170.988002   
                             opexCap         [1e9 Euro/a]                 0.007791   
Hydrogen demand              operation       [GW$_{H_{2},LHV}$*h/a]   11007.056882   
                                             [GW$_{H_{2},LHV}$*h]     11007.056882   
PV                           NPVcontribution [1e9 Euro]                   5.605296   
                             TAC             [1e9 Euro/a]                 1.909967   
                             capacity        [GW$_{el}$]                 25.848365   
                             capexCap        [1e9 Euro/a]                 1.573938   
                             operation       [GW$_{el}$*h/a]          23816.214088   
                                             [GW$_{el}$*h]            23816.214088   
                             opexCap         [1e9 Euro/a]                 0.336029   
Steel demand                 operation       [tons/a]                   209.754434   
Steel supply                 operation       [tons/a]                    136.06071   
windoffshore                 NPVcontribution [1e9 Euro]                   7.302271   
                             TAC             [1e9 Euro/a]                 2.488199   
                             capacity        [GW$_{el}$]                       4.0   
                             capexCap        [1e9 Euro/a]                 2.304199   
                             commissioning   [GW$_{el}$]                       4.0   
                             decommissioning [GW$_{el}$]                       4.0   
                             invest          [1e9 Euro]                        9.2   
                             operation       [GW$_{el}$*h/a]          14131.818772   
                                             [GW$_{el}$*h]            14131.818772   
                             opexCap         [1e9 Euro/a]                    0.184   
windoffshore_copper_scrap    operation       [tons/a]                        10.54   
windoffshore_steel_scrap     operation       [tons/a]                        10.54   
windonshore                  NPVcontribution [1e9 Euro]                  57.552973   
                             TAC             [1e9 Euro/a]                19.610786   
                             capacity        [GW$_{el}$]                 65.918145   
                             capexCap        [1e9 Euro/a]                18.160587   
                             commissioning   [GW$_{el}$]                 65.918145   
                             decommissioning [GW$_{el}$]                 65.918145   
                             invest          [1e9 Euro]                  72.509959   
                            

In [40]:
summary = esM.getOptimizationSummary("SourceSinkModel", ip=2025, outputLevel=0)

# Inspect index levels to confirm structure
print(summary.index.names)


['Component', 'Property', 'Unit']


In [ ]:
summary_lithium_demand = summary.loc[("Windoffshore_steel_scrap demand",)]

: 

In [ ]:
summary_lithium_demand

: 

In [ ]:
decommis = esM.pyM.decommis_SourceSink
for (loc2, compName, ip2), val in decommis.items():
    if ip2 == 1 and "windonshore" in compName:
        print(f"{compName} @ {loc2}, ip={ip2} → {float(val())}")


: 

In [ ]:
import pyomo.environ as pyomo

: 

In [ ]:
print(list(esM.pyM.component_objects(pyomo.Var)))


: 

In [ ]:
for var in esM.pyM.component_objects(pyomo.Var):
    print(var.name)


: 

In [ ]:
decommis = esM.pyM.decommis_srcSnk
for (loc2, compName, ip2), val in decommis.items():
    if ip2 == 1 and "windonshore" in compName:
        print(f"{compName} @ {loc2}, ip={ip2} → {float(val()):.4f}")


: 

In [ ]:
"windoffshore_copper_scrap" in esM.componentModelingDict["SourceSinkModel"].componentsDict
list(esM.componentModelingDict["SourceSinkModel"].componentsDict.keys())





: 

In [ ]:
for compName, comp in esM.componentModelingDict["SourceSinkModel"].componentsDict.items():
    if "lithium" in comp.commodity.lower():
        print(f"{compName} ({comp.commodity})")


: 

Plot installed capacities

In [ ]:
fig, ax = fn.plotLocationalColorMap(
    esM, "Wind (offshore)", locFilePath, "index", perArea=False
)

: 

In [ ]:
fig, ax = fn.plotLocationalColorMap(
    esM, "Wind (onshore)", locFilePath, "index", perArea=False
)

: 

In [ ]:
fig, ax = fn.plotLocationalColorMap(esM, "PV", locFilePath, "index", perArea=False)

: 

Plot operation time series (either one or two dimensional)

In [ ]:
fig, ax = fn.plotOperationColorMap(esM, "Electricity demand", "GermanyRegion")

: 

In [ ]:
fig, ax = fn.plotOperationColorMap(esM, "Electricity demand", "FranceRegion", ip = 2020)

: 

### Conversion

Show optimization summary

In [ ]:
esM.getOptimizationSummary("ConversionModel", outputLevel=2)

: 

In [ ]:
fig, ax = fn.plotLocationalColorMap(
    esM, "Electrolyzer", locFilePath, "index", perArea=False
)

: 

In [ ]:
fig, ax = fn.plotOperationColorMap(esM, "New CCGT plants (biogas)", "GermanyRegion")

: 

In [ ]:
fig, ax = fn.plotOperationColorMap(esM, "New CCGT plants (biogas)", "FranceRegion")

: 

### Storage

Show optimization summary

In [ ]:
esM.getOptimizationSummary("StorageModel", outputLevel=2)

: 

In [ ]:
fig, ax = fn.plotOperationColorMap(
    esM,
    "Li-ion batteries",
    "FranceRegion",
    variableName="stateOfChargeOperationVariablesOptimum",
)

: 

In [ ]:
fig, ax = fn.plotOperationColorMap(
    esM,
    "Pumped hydro storage",
    "FranceRegion",
    variableName="stateOfChargeOperationVariablesOptimum",
)

: 

In [ ]:
fig, ax = fn.plotOperationColorMap(
    esM,
    "Salt caverns (biogas)",
    "FranceRegion",
    variableName="stateOfChargeOperationVariablesOptimum",
)

: 

In [ ]:
fig, ax = fn.plotOperationColorMap(
    esM,
    "Salt caverns (hydrogen)",
    "FranceRegion",
    variableName="stateOfChargeOperationVariablesOptimum",
)

: 

## Transmission

Show optimization summary

In [ ]:
esM.getOptimizationSummary("TransmissionModel", outputLevel=2)

: 

In [ ]:
esM.getOptimizationSummary("TransmissionModel", outputLevel=2).loc[
    "Pipelines (hydrogen)"
]

: 

Check that the shared capacity of the pipelines are not exceeded

In [ ]:
df = esM.componentModelingDict["TransmissionModel"].capacityVariablesOptimum
df.loc["Pipelines (biogas)"] + df.loc["Pipelines (hydrogen)"]

: 

: 

: 

: 

: 